# Does knowing the floor area make the predictions better?

The main notebook's best model is still over £100,000 out on average. The obvious thing it's missing is size. To those models a small three-bed detached house and a huge seven-bed one in the same postcode district look exactly the same.

So this notebook tests that, and only that:

- **Model A**: property type, freehold or leasehold, postcode district, month of sale.
- **Model B**: exactly the same, plus total floor area in m².

Both Random Forests, tuned the same way, tested on the same sales. Whatever gap there is comes from the floor area.

The floor areas come from UCL's linked dataset (Chi et al., 2021; Chi et al., 2025), which already matches Land Registry sales to Energy Performance Certificates. Writing my own address matcher was not something I wanted to take on.

Three limits to keep in mind:
- The UCL file stops at 31 October 2024, so this runs on 2021-2024, not the 2025 test period of the main notebook. Don't compare the two notebooks' numbers. A against B is the comparison here.
- The certificates were linked after the fact. I can't check each one was issued before its sale, so a few floor areas might include later extensions.
- The file has habitable rooms too (not bedrooms). The models don't use it (one change at a time), but section 10 uses it for the price tables.

Sections: 0 setup · 1 the UCL file · 2 pull out the three boroughs · 3 clean · 4 features and split · 5 models · 6 tuning · 7 final test · 8 where floor area helps · 9 three checks · 10 what homes of each size sold for · 11 save

## 0. Setup

In [1]:
import hashlib
import os
import shutil
import sys
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error,
                             r2_score, mean_absolute_percentage_error)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42          # same seed as the main notebook
RESULTS_DIR = "results"

UCL_DIR = "data/ucl"                                   # UCL download + my extract
UCL_ZIP = f"{UCL_DIR}/hpm_la_2024.zip"                 # 1.24 GB
UCL_URL = "https://data.london.gov.uk/download/epo9w/e4a4f81e-d922-4d14-b6f7-9a4d8c09e9bc/hpm_la_2024.zip"
EXTRACT = f"{UCL_DIR}/three_boroughs_2021_2024.csv"    # the ~42k rows I actually use

# the London Datastore gives python's default client name a 403
USER_AGENT = "Mozilla/5.0 (compatible; house-price-ml research notebook)"

BOROUGHS = {"E09000006": "Bromley", "E09000008": "Croydon", "E09000011": "Greenwich"}
YEARS = ["2021", "2022", "2023", "2024"]

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(UCL_DIR, exist_ok=True)

print("python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", sklearn.__version__)

python 3.12.3 | pandas 3.1.1 | numpy 2.5.4 | sklearn 1.9.1


## 1. The UCL file
One 1.24 GB zip with every linked sale in England and Wales since 1995. The notebook downloads it if it isn't in `data/ucl/` already. Slow, but only once.

For a while this download didn't work. The London Datastore refuses Python's default client name with a 403, and nobody noticed because on the earlier runs the zip had been fetched with curl first. It only showed up when I ran the notebook in a completely empty folder. `download()` below now sends its own client name, and the empty-folder run went through on 18 November 2026.

If it still fails, get **House price per square metre dataset 2024** by hand from https://data.london.gov.uk/dataset/house-price-per-square-metre-in-england-and-wales and put `hpm_la_2024.zip` in `data/ucl/`.

The field guide was written for an older version of the file, so I print the column names before trusting any of them. The local authority column turned out to be `lad23cd`, and the dates are written 2024-03-11, not 11/3/2014 like the guide shows.

In [2]:
def download(url, path):
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    # .part first, so a cut-off download never looks finished
    with urllib.request.urlopen(request) as response, open(path + ".part", "wb") as f:
        shutil.copyfileobj(response, f, length=1024 * 1024)
    os.replace(path + ".part", path)

# extract already there -> big file not needed
need_big_file = not os.path.exists(EXTRACT)

if need_big_file and not os.path.exists(UCL_ZIP):
    print("downloading the UCL file (1.24 GB). on a slow connection this takes a long time...")
    download(UCL_URL, UCL_ZIP)

In [3]:
# columns I need, named as in UCL's field guide.
# not priceper: it's price / floor area, i.e. the answer
NEEDED = ["transactionid", "year", "dateoftransfer", "price", "propertytype",
          "duration", "postcode", "tfarea", "numberrooms", "classt"]

if need_big_file:
    with zipfile.ZipFile(UCL_ZIP) as z:
        csv_names = [name for name in z.namelist() if name.lower().endswith(".csv")]
        assert csv_names, f"no csv files inside {UCL_ZIP}. it holds: {z.namelist()[:10]}"
        with z.open(csv_names[0]) as f:
            peek = pd.read_csv(f, nrows=5, dtype=str)

    # names only. this used to display the first rows too, took that out
    print(len(csv_names), "csv file(s) in the zip, starting with:", csv_names[:3])
    print("columns:", list(peek.columns))

    # ignore case, TFAREA vs tfarea
    real_name = {col.lower(): col for col in peek.columns}
    missing = [col for col in NEEDED if col not in real_name]
    assert not missing, f"the UCL file has no column called {missing}. its columns are: {list(peek.columns)}"

    # lad21cd in older versions, lad23cd in this one
    la_columns = [col for col in peek.columns if col.lower().startswith("lad")]
    assert la_columns, f"no local authority column found. the columns are: {list(peek.columns)}"
    la_col = la_columns[0]
    print("local authority column:", la_col)
else:
    print("extract already made, so the big UCL file isn't read again:", EXTRACT)

extract already made, so the big UCL file isn't read again: data/ucl/three_boroughs_2021_2024.csv


## 2. Pull out the three boroughs
22 million rows is far more than I need. I read it half a million rows at a time, keep Bromley, Croydon and Greenwich from 2021 on, and save that as a small extract (about 42,000 rows). After the first run everything starts from the extract.

Then the notebook fingerprints the zip and the extract (SHA-256) into `results/floor_area_inputs.csv`. If you re-run and your zip's fingerprint is different, UCL have changed the file and your numbers may move. Mine matched between a curl download on 17 November and the notebook's own download on 18 November.

In [4]:
if need_big_file:
    columns = [real_name[col] for col in NEEDED] + [la_col]
    kept = []      # ~22M rows, half a million at a time
    with zipfile.ZipFile(UCL_ZIP) as z:
        for name in csv_names:
            with z.open(name) as f:
                for chunk in pd.read_csv(f, usecols=columns, dtype=str, chunksize=500_000):
                    ours = chunk[la_col].isin(BOROUGHS) & chunk[real_name["year"]].isin(YEARS)
                    kept.append(chunk[ours])

    extract = pd.concat(kept, ignore_index=True)
    extract = extract.rename(columns={real_name[col]: col for col in NEEDED})   # back to the field-guide names
    extract = extract.rename(columns={la_col: "la_code"})
    extract.to_csv(EXTRACT, index=False)

linked = pd.read_csv(EXTRACT, dtype=str)
print(len(linked), "linked sales for the three boroughs, 2021-2024")
print("columns:", list(linked.columns))

41871 linked sales for the three boroughs, 2021-2024
columns: ['year', 'dateoftransfer', 'propertytype', 'duration', 'price', 'postcode', 'la_code', 'transactionid', 'tfarea', 'numberrooms', 'classt']


In [5]:
def sha256(path):
    # 1 MB at a time, the zip doesn't need to fit in memory
    fingerprint = hashlib.sha256()
    with open(path, "rb") as f:
        while piece := f.read(1024 * 1024):
            fingerprint.update(piece)
    return fingerprint.hexdigest()

# one row per input file, zip skipped if it's been deleted.
# "file date" = file timestamp, so the download day for the zip. careful with the time of day though:
# the clock on the machine I ran this on jumped about (the zip showed a later time than the extract
# made from it). only the date gets saved, and that was right
files = []
for path in [UCL_ZIP, EXTRACT]:
    if os.path.exists(path):
        files.append({"file": path,
                      "size (bytes)": os.path.getsize(path),
                      "file date": pd.Timestamp(os.path.getmtime(path), unit="s").strftime("%Y-%m-%d"),
                      "sha256": sha256(path)})
inputs = pd.DataFrame(files)
inputs

,file,size (bytes),file date,sha256
0,data/ucl/hpm_la_2024.zip,1336477789,2026-11-18,73e47dbfca327ac54655d5329d4272e25e89407fbb43c3...
1,data/ucl/three_boroughs_2021_2024.csv,3883697,2026-11-18,3da52d17eeb1c7737a461ef09b20b8e308a30e16e98fc9...


## 3. Cleaning
UCL leave the tidying to whoever uses the file, so:

1. has a price and a readable sale date
2. house or flat, freehold or leasehold
3. valid postcode
4. each sale once. Exact copies are cut to one row. A sale that shows up with two *different* floor areas is dropped, since I can't tell which is right
5. floor area between 10 and 1,000 m²

Rules 1-4 remove nothing at all on this file. I kept them anyway: they cost nothing, and a later version of the file might not be this tidy.

Rule 5 is the one that actually bites, and not entirely fairly. It takes out 12 sales. Eight are flats at 0 or 3 m², obviously wrong. Of the four over 1,000 m², a £450k semi at 1,216 m² is clearly a typo, but the other three are Bromley detached houses at £5.3m-£15.4m (the biggest 1,647 m²), and those could well be real. I left the limits where they are, it's 12 sales out of nearly 42,000, but the very top of the market is a little thinner here than it really is.

No rule looks at the price.

In [6]:
POSTCODE_RE = r"^[A-Z]{1,2}[0-9][A-Z0-9]? [0-9][A-Z]{2}$"   # e.g. "BR3 3JW"
MIN_AREA, MAX_AREA = 10, 1000                                # m2

def read_dates(text):
    # guide says 11/3/2014, Land Registry writes 2014-03-11. try each, keep whichever reads most rows
    # (this file turned out to be all 2014-03-11)
    best = None
    for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%Y-%m-%d %H:%M"]:
        dates = pd.to_datetime(text, format=fmt, errors="coerce")
        if best is None or dates.notna().sum() > best.notna().sum():
            best = dates
    return best

steps = []

def log_step(name, d):
    steps.append({"step": name, "rows": len(d)})

# rules 1-4 remove nothing on this file. kept anyway
df = linked.copy()
log_step("linked sales, three boroughs, 2021-2024", df)

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["floor_area"] = pd.to_numeric(df["tfarea"], errors="coerce")
df["date"] = read_dates(df["dateoftransfer"])

df = df[(df["price"] > 0) & df["date"].notna()]
log_step("has a price and a sale date", df)

df = df[df["propertytype"].isin(["D", "S", "T", "F"]) & df["duration"].isin(["F", "L"])]
log_step("house or flat, freehold or leasehold", df)

df = df[df["postcode"].fillna("").str.match(POSTCODE_RE)]
log_step("valid postcode", df)

df = df.drop_duplicates(["transactionid", "floor_area"])
log_step("exact copies of a sale removed", df)

# still there twice = two different floor areas. can't tell which is right, drop both
df = df[~df["transactionid"].duplicated(keep=False)]
log_step("sales with two different floor areas removed", df)

outside = df[~df["floor_area"].between(MIN_AREA, MAX_AREA)]     # set aside for the next cell
df = df[df["floor_area"].between(MIN_AREA, MAX_AREA)]
log_step(f"floor area between {MIN_AREA} and {MAX_AREA} m2", df)

data_quality = pd.DataFrame(steps)
data_quality["removed"] = (-data_quality["rows"].diff()).fillna(0).astype(int)
data_quality

,step,rows,removed
0,"linked sales, three boroughs, 2021-2024",41871,0
1,has a price and a sale date,41871,0
2,"house or flat, freehold or leasehold",41871,0
3,valid postcode,41871,0
4,exact copies of a sale removed,41871,0
5,sales with two different floor areas removed,41871,0
6,floor area between 10 and 1000 m2,41859,12


In [7]:
# what rule 5 threw out. MIN_AREA / MAX_AREA above if this looks wrong
print(outside["floor_area"].isna().sum(), "sales with no floor area,",
      (outside["floor_area"] < MIN_AREA).sum(), f"under {MIN_AREA} m2,",
      (outside["floor_area"] > MAX_AREA).sum(), f"over {MAX_AREA} m2")

# all twelve, biggest first, with the borough. 3 of the 4 over 1000 m2 are £5m+ Bromley detached, probably real.
# the £450k semi at 1,216 m2 isn't. (postcodes left out on purpose)
(outside.assign(borough=outside["la_code"].map(BOROUGHS))
 .sort_values("floor_area", ascending=False)
 [["dateoftransfer", "borough", "propertytype", "price", "floor_area"]].head(12))

0 sales with no floor area, 8 under 10 m2, 4 over 1000 m2


,dateoftransfer,borough,propertytype,price,floor_area
5335,2022-04-05,Bromley,D,15375000,1647.0
5895,2022-10-26,Bromley,D,5300000,1417.0
5669,2021-06-10,Bromley,D,6000000,1281.0
25020,2022-08-05,Croydon,S,450000,1216.0
18467,2022-02-11,Croydon,F,210000,3.0
15817,2022-04-06,Bromley,F,275000,0.0
20148,2023-11-24,Croydon,F,144000,0.0
19835,2022-11-18,Croydon,F,134000,0.0
20567,2022-10-14,Croydon,F,162500,0.0
39928,2022-03-11,Greenwich,F,428000,0.0


In [8]:
assert df["transactionid"].is_unique
assert set(df["la_code"]) <= set(BOROUGHS)
assert (df["date"].dt.year.astype(str) == df["year"]).all()      # my dates agree with UCL's year
assert df["date"].max() <= pd.Timestamp("2024-10-31")           # file stops here

print(df[["price", "floor_area"]].describe().round(0))

           price  floor_area
count    41859.0     41859.0
mean    516953.0        92.0
std     286980.0        47.0
min      80000.0        14.0
25%     350000.0        64.0
50%     457000.0        83.0
75%     610000.0       107.0
max    6400000.0       892.0


## 4. Features and split
- **Model A**: property type, tenure, postcode district, month of sale.
- **Model B**: same four plus floor area.

Same pattern as the main notebook, moved back to fit the UCL file: learn on 2021-22, tune on 2023, test once on Jan-Oct 2024.

The months aren't even, and it's worth seeing (the monthly counts print below). The file was put together in December 2024 and sales take a while to reach the Land Registry, so October 2024 has only 244 sales after cleaning, against about 600 a month earlier in 2024. 2021 is lumpy the other way: March and June are huge (2,022 and 3,206) and July drops to 431, which lines up with the stamp-duty holiday deadlines. That's all training data, and month of sale is one of the features.

In [9]:
df = df.rename(columns={"propertytype": "property_type"})
df["borough"] = df["la_code"].map(BOROUGHS)
df["postcode_district"] = df["postcode"].str.split().str[0]   # "BR3 3JW" -> "BR3"
df["sale_month"] = df["date"].dt.month

CATEGORIES = ["property_type", "duration", "postcode_district", "sale_month"]   # what Model A is told
NUMBERS = ["floor_area"]                                                         # the one extra thing Model B gets

# are the codes right? Bromley -> BR, Croydon -> CR, Greenwich -> SE
print(df.groupby("borough")["postcode_district"].agg(lambda districts: districts.mode()[0]))

# sales per month. Oct 2024 is thin (late registrations), 2021 jumps around (stamp duty holiday)
print(df["date"].dt.to_period("M").value_counts().sort_index().to_string())

train = df[df["date"].dt.year <= 2022]
val   = df[df["date"].dt.year == 2023]
test  = df[df["date"].dt.year == 2024]

# no overlap, every row in exactly one split
assert train["date"].max() < val["date"].min() < val["date"].max() < test["date"].min()
assert len(train) + len(val) + len(test) == len(df)

split_sizes = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "period": ["2021-2022", "2023", "Jan-Oct 2024"],
    "rows": [len(train), len(val), len(test)],
    "median price": [part["price"].median() for part in (train, val, test)],
    "median floor area (m2)": [part["floor_area"].median() for part in (train, val, test)]})
split_sizes

borough
Bromley       BR3
Croydon       CR0
Greenwich    SE18
Name: postcode_district, dtype: str
date
2021-01    1083
2021-02    1289
2021-03    2022
2021-04    1170
2021-05     990
2021-06    3206
2021-07     431
2021-08     834
2021-09    1603
2021-10     697
2021-11     939
2021-12    1001
2022-01     794
2022-02     939
2022-03    1012
2022-04     929
2022-05     925
2022-06    1026
2022-07    1175
2022-08    1180
2022-09    1181
2022-10    1126
2022-11    1124
2022-12    1010
2023-01     728
2023-02     714
2023-03     774
2023-04     509
2023-05     595
2023-06     685
2023-07     755
2023-08     866
2023-09     808
2023-10     763
2023-11     719
2023-12     647
2024-01     613
2024-02     602
2024-03     637
2024-04     585
2024-05     691
2024-06     571
2024-07     562
2024-08     613
2024-09     492
2024-10     244
Freq: M


,split,period,rows,median price,median floor area (m2)
0,train,2021-2022,27686,457000.0,83.0
1,validation,2023,8563,453000.0,82.0
2,test,Jan-Oct 2024,5610,462500.0,83.0


## 5. Models
Same recipe as the main notebook. The only new bit is that floor area is a number, so it goes straight in next to the one-hot categories.

In [10]:
def make_model(model, categories, numbers=()):
    # one-hot the categories, numbers pass straight through, learn log(price), predict back in £
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    prep = ColumnTransformer([("onehot", onehot, categories),
                              ("numbers", "passthrough", list(numbers))])
    pipe = Pipeline([("prep", prep), ("model", model)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)

def scores(y_true, y_pred):
    # same four metrics as the main notebook, MAE is the one I care about
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": root_mean_squared_error(y_true, y_pred),
            "R2": r2_score(y_true, y_pred),
            "MAPE %": 100 * mean_absolute_percentage_error(y_true, y_pred)}

def rf(**params):
    # 300 trees, like the main notebook
    return RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, **params)

## 6. Tuning
Same eight settings as the main notebook, tried on 2023, lowest error wins.

A and B are tuned separately. A forest with floor area to split on may want different settings, and forcing Model A's onto it would tilt things against it. They did end up different (A picked `sqrt`, B picked all features), so section 9 checks whether that matters.

In [11]:
def tune(numbers):
    # 8 settings, fit on 2021-22, score on 2023
    features = CATEGORIES + numbers
    rows = []
    for leaf in [1, 5, 10, 20]:
        for max_feat in [1.0, "sqrt"]:
            m = make_model(rf(min_samples_leaf=leaf, max_features=max_feat), CATEGORIES, numbers)
            m.fit(train[features], train["price"])
            rows.append({"min_samples_leaf": leaf,
                         "max_features": max_feat,
                         "train MAE": mean_absolute_error(train["price"], m.predict(train[features])),
                         "val MAE": mean_absolute_error(val["price"], m.predict(val[features]))})
    table = pd.DataFrame(rows)
    best = table.loc[table["val MAE"].idxmin()]
    best_params = {"min_samples_leaf": int(best["min_samples_leaf"]), "max_features": best["max_features"]}
    return table, best_params

tuning_a, params_a = tune([])
tuning_b, params_b = tune(NUMBERS)
# B with leaf=1 and all features is a nice warning: £37,683 on train but £88,163 on 2023. overfit.
# the leaf=10 it actually picked gets £83,113 on 2023

print("Model A settings:", params_a)
print("Model B settings:", params_b)

tuning = pd.concat({"A, without floor area": tuning_a, "B, with floor area": tuning_b}, names=["model"])
tuning = tuning.reset_index(level="model").reset_index(drop=True)
tuning.round(0)

Model A settings: {'min_samples_leaf': 10, 'max_features': 'sqrt'}
Model B settings: {'min_samples_leaf': 10, 'max_features': 1.0}


,model,min_samples_leaf,max_features,train MAE,val MAE
0,"A, without floor area",1,1.0,111648.0,122707.0
1,"A, without floor area",1,sqrt,111656.0,122806.0
2,"A, without floor area",5,1.0,113056.0,121983.0
3,"A, without floor area",5,sqrt,113796.0,120763.0
4,"A, without floor area",10,1.0,114039.0,121427.0
5,"A, without floor area",10,sqrt,115144.0,120356.0
6,"A, without floor area",20,1.0,115400.0,121269.0
7,"A, without floor area",20,sqrt,116570.0,120444.0
8,"B, with floor area",1,1.0,37683.0,88163.0
9,"B, with floor area",1,sqrt,38500.0,91254.0


## 7. Final test
Both refitted on 2021-2023 with their own settings, scored once on Jan-Oct 2024. The last row is A minus B, so a positive MAE difference means floor area helped.

In [12]:
# refit on 2021-2023, score once on 2024
train_val = pd.concat([train, val])
y = test["price"].to_numpy()

model_a = make_model(rf(**params_a), CATEGORIES)
model_a.fit(train_val[CATEGORIES], train_val["price"])
pred_a = model_a.predict(test[CATEGORIES])

model_b = make_model(rf(**params_b), CATEGORIES, NUMBERS)
model_b.fit(train_val[CATEGORIES + NUMBERS], train_val["price"])
pred_b = model_b.predict(test[CATEGORIES + NUMBERS])

comparison = pd.DataFrame({"Model A, without floor area": scores(y, pred_a),
                           "Model B, with floor area": scores(y, pred_b)}).T
comparison.loc["Difference (A - B)"] = comparison.iloc[0] - comparison.iloc[1]
comparison.round(3)

,MAE,RMSE,R2,MAPE %
"Model A, without floor area",118043.875,210872.922,0.474,24.065
"Model B, with floor area",80226.513,135532.224,0.783,16.759
Difference (A - B),37817.362,75340.697,-0.309,7.306


**How firm is it?** Same check as the main notebook: redraw the test sales 1,000 times and watch the MAE gap move. If the whole 95% range is above zero, the gain is bigger than the sampling noise.

It isn't proof in general, though. Nearby homes tend to be wrong in the same direction and this ignores that, and it says nothing about later years.

In [13]:
rng = np.random.default_rng(RANDOM_STATE)
err_a = np.abs(y - pred_a)
err_b = np.abs(y - pred_b)

boot_idx = rng.integers(0, len(y), size=(1000, len(y)))
gains = (err_a - err_b)[boot_idx].mean(axis=1)
low, high = np.percentile(gains, [2.5, 97.5])

print(f"MAE without floor area minus MAE with it: £{(err_a - err_b).mean():+,.0f}")
print(f"95% range: £{low:+,.0f} to £{high:+,.0f}")

MAE without floor area minus MAE with it: £+37,817
95% range: £+34,900 to £+40,715


## 8. Where floor area helps
In the main notebook the worst misses were the expensive homes and the detached houses. If size is what's missing, those should improve most.

In [14]:
results = test.assign(err_a=err_a, err_b=err_b)
results["type"] = results["property_type"].map({"D": "Detached", "S": "Semi", "T": "Terraced", "F": "Flat"})
top_tenth = results["price"] >= results["price"].quantile(0.9)
results["price band"] = np.where(top_tenth, "top 10% by price", "other 90%")

def breakdown(col):
    # n, typical price, both models' MAE per group
    table = results.groupby(col).agg(sales=("price", "size"),
                                     median_price=("price", "median"),
                                     mae_without=("err_a", "mean"),
                                     mae_with=("err_b", "mean"))
    table["gain"] = table["mae_without"] - table["mae_with"]
    return table.round(0)

by_group = pd.concat({"property type": breakdown("type"),
                      "price band": breakdown("price band"),
                      "borough": breakdown("borough")})
by_group

sales  median_price  mae_without  mae_with  \
property type Detached            645      780000.0     248437.0  160683.0   
              Flat               1980      320000.0      87013.0   59450.0   
              Semi               1235      570000.0     129450.0   86718.0   
              Terraced           1750      470000.0      97045.0   69499.0   
price band    other 90%          5043      440000.0      89570.0   63657.0   
              top 10% by price    567      970200.0     371300.0  227602.0   
borough       Bromley            2331      507500.0     135791.0   90658.0   
              Croydon            1909      422000.0      87034.0   62642.0   
              Greenwich          1370      460000.0     131058.0   86979.0   

                                    gain  
property type Detached           87754.0  
              Flat               27563.0  
              Semi               42732.0  
              Terraced           27546.0  
price band    other 90%          25913.0  
              top 10% by price  143698.0  
borough       Bromley            45132.0  
              Croydon            24392.0  
              Greenwich          44079.0

## 9. Three checks
**Is it just the tuning?** A and B got different settings, so they aren't literally the same forest plus one column. This reruns B with A's settings.

In [15]:
model_b_same = make_model(rf(**params_a), CATEGORIES, NUMBERS)
model_b_same.fit(train_val[CATEGORIES + NUMBERS], train_val["price"])
err_b_same = np.abs(y - model_b_same.predict(test[CATEGORIES + NUMBERS]))

settings_check = pd.DataFrame({
    "settings": [str(params_a), str(params_b), str(params_a)],
    "MAE": [err_a.mean(), err_b.mean(), err_b_same.mean()]},
    index=["Model A", "Model B, its own settings", "Model B, Model A's settings"])
settings_check["gain over Model A"] = err_a.mean() - settings_check["MAE"]
settings_check.round(0)

,settings,MAE,gain over Model A
Model A,"{'min_samples_leaf': 10, 'max_features': 'sqrt'}",118044.0,0.0
"Model B, its own settings","{'min_samples_leaf': 10, 'max_features': 1.0}",80227.0,37817.0
"Model B, Model A's settings","{'min_samples_leaf': 10, 'max_features': 'sqrt'}",83165.0,34879.0


**Does it hold on the cleanest matches?** UCL mark a sale `11` if it matched exactly one certificate and `12` if it matched more than one. This scores the 11s on their own.

In [16]:
one_to_one = (pd.to_numeric(test["classt"], errors="coerce") == 11).to_numpy()
print(f"{one_to_one.mean():.0%} of the test sales are one-to-one matches")

match_check = pd.DataFrame({
    "sales": [len(y), one_to_one.sum()],
    "MAE without floor area": [err_a.mean(), err_a[one_to_one].mean()],
    "MAE with floor area": [err_b.mean(), err_b[one_to_one].mean()]},
    index=["all test sales", "one-to-one matches only"])
match_check["gain"] = match_check["MAE without floor area"] - match_check["MAE with floor area"]
match_check.round(0)

79% of the test sales are one-to-one matches


,sales,MAE without floor area,MAE with floor area,gain
all test sales,5610,118044.0,80227.0,37817.0
one-to-one matches only,4434,118012.0,80373.0,37639.0


**How many sales have a floor area at all?** Only sales UCL could match to a certificate are in their file. The main notebook's cleaned data has every standard sale in the three boroughs for 2024, so I can count how many from January to October turn up here, and whether the missing ones look different.

About 62% do, and flats are under-represented among them. One loose end: the test set has 5,610 sales, but only 5,553 of them are found in the main notebook's file. I haven't chased down why.

Needs the file the main notebook writes. Skipped if it isn't there.

In [17]:
MAIN_FILE = "data/processed/three_boroughs_2024_2025.csv"

def plain_id(ids):
    # IDs come wrapped in {}, strip so the two files match
    return ids.str.strip("{}").str.upper()

if os.path.exists(MAIN_FILE):
    main = pd.read_csv(MAIN_FILE, parse_dates=["date"])
    main = main[main["date"] <= "2024-10-31"]
    main["has floor area"] = plain_id(main["transaction_id"]).isin(set(plain_id(test["transactionid"])))

    print(f"{main['has floor area'].mean():.0%} of the {len(main):,} sales from January to October 2024 have a usable floor area")
    coverage = main.groupby("has floor area").agg(sales=("price", "size"),
                                                  median_price=("price", "median"),
                                                  share_flats=("property_type", lambda t: (t == "F").mean()))
    display(coverage.round(2))
else:
    print("skipped: run house_prices.ipynb first to create", MAIN_FILE)

62% of the 8,890 sales from January to October 2024 have a usable floor area


,sales,median_price,share_flats
has floor area,,,
False,3337,450000.0,0.46
True,5553,465000.0,0.35


## 10. What homes of each size sold for
No model here, just what homes actually sold for, in the kind of table a buyer or a surveyor would recognise.

Three tables from the cleaned sales, 2021 to October 2024:
- **by floor area**: typical price and price per m², per property type and size band
- **by habitable rooms**: typical price and floor area. Habitable rooms are bedrooms plus living and dining rooms, so a three-bed house is usually 4 or 5
- **by borough**: typical price, floor area and price per m²

"Typical" = median. Groups with fewer than 20 sales are dropped, because a median of a handful of sales jumps about too much. Prices are rounded to the nearest £1,000. Price per m² is rounded to £1 instead: rounding a figure of about £5,000 to the nearest £1,000 throws away most of it.

These tables started life as a one-off script outside the notebook. When I moved them in here I checked the CSVs against that script's output, and they were byte for byte the same.

Past sales, not valuations. And each table looks at one thing at a time: bigger homes also tend to be in pricier streets, so some of the "size effect" here is really location. Pulling those apart is Model B's job.

Price per m² is only used here to describe the sales. It never goes into a model, because it comes from the price.

In [18]:
TYPE_NAMES = {"D": "Detached", "S": "Semi", "T": "Terraced", "F": "Flat"}
MIN_SALES = 20      # smaller groups get dropped

sales = df.copy()
sales["type"] = pd.Categorical(sales["property_type"].map(TYPE_NAMES),
                               categories=list(TYPE_NAMES.values()), ordered=True)   # keeps this order in the tables
sales["price_per_m2"] = sales["price"] / sales["floor_area"]

# bands include their lower edge: 75 m2 -> 75-99, 74.6 -> 50-74
sales["size_band"] = pd.cut(sales["floor_area"], bins=[0, 50, 75, 100, 150, 200, 300, np.inf], right=False,
                            labels=["under 50", "50-74", "75-99", "100-149", "150-199", "200-299", "300+"])

NICE_NAMES = {"median_price": "median price",
              "median_price_per_m2": "median price per m2",
              "median_area": "median floor area (m2)"}

def finish(table, index_names):
    # drop small groups (and say which), readable names, round: £1,000 / £1 per m2 / 0.1 m2
    for group, n in table.loc[table["sales"] < MIN_SALES, "sales"].items():
        print("left out, too few sales:", *group, f"({n})")
    table = table[table["sales"] >= MIN_SALES].rename(columns=NICE_NAMES).rename_axis(index_names)
    table = table.round({"median price": -3, "median price per m2": 0, "median floor area (m2)": 1})
    whole_pounds = [col for col in ["median price", "median price per m2"] if col in table]
    return table.astype({col: int for col in whole_pounds})

by_size = sales.groupby(["type", "size_band"], observed=True).agg(
    sales=("price", "size"),
    median_price=("price", "median"),
    median_price_per_m2=("price_per_m2", "median"))
by_size = finish(by_size, ["property type", "floor area band (m2)"])
by_size

left out, too few sales: Detached under 50 (8)
left out, too few sales: Terraced 300+ (8)
left out, too few sales: Flat 200-299 (16)
left out, too few sales: Flat 300+ (6)


sales  median price  median price per m2
property type floor area band (m2)                                          
Detached      50-74                   153        485000                 7170
              75-99                   632        586000                 6620
              100-149                1886        729000                 5955
              150-199                1098        950000                 5587
              200-299                 625       1250000                 5395
              300+                    202       1959000                 5128
Semi          under 50                 59        346000                 7683
              50-74                  1045        427000                 6469
              75-99                  3128        510000                 5791
              100-149                3618        630000                 5350
              150-199                 822        800000                 4870
              200-299                 175       1050000                 4861
              300+                     25       1680000                 4771
Terraced      under 50                194        294000                 6801
              50-74                  2801        385000                 5909
              75-99                  5558        450000                 5275
              100-149                3275        570000                 4955
              150-199                 297        770000                 4759
              200-299                  55       1150000                 5197
Flat          under 50               3389        245000                 5778
              50-74                  8640        330000                 5400
              75-99                  3269        440000                 5319
              100-149                 791        585000                 5069
              150-199                  84        780000                 4741

In [19]:
rooms = pd.to_numeric(sales["numberrooms"], errors="coerce")
print(rooms.isna().sum(), "sales have no room count, so they are left out of this table only")
print((rooms > 20).sum(), "sales list more than 20 rooms, which look like typing errors (they land in the 9+ group)")

print(rooms.describe().round(1))     # two sales say 51 rooms. typos surely, they end up in 9+
print((rooms == 51).sum(), "sales list 51 rooms")

with_rooms = sales[rooms.notna()].copy()
with_rooms["rooms"] = rooms[rooms.notna()].astype(int).clip(upper=9).astype(str).replace("9", "9+")

by_rooms = with_rooms.groupby(["type", "rooms"], observed=True).agg(
    sales=("price", "size"),
    median_price=("price", "median"),
    median_area=("floor_area", "median"))
by_rooms = finish(by_rooms, ["property type", "habitable rooms"])
by_rooms

5980 sales have no room count, so they are left out of this table only
2 sales list more than 20 rooms, which look like typing errors (they land in the 9+ group)
count    35879.0
mean         4.4
std          1.7
min          1.0
25%          3.0
50%          4.0
75%          5.0
max         51.0
Name: numberrooms, dtype: float64
2 sales list 51 rooms
left out, too few sales: Detached 1 (7)
left out, too few sales: Detached 2 (8)
left out, too few sales: Semi 1 (7)
left out, too few sales: Terraced 1 (6)
left out, too few sales: Flat 7 (17)
left out, too few sales: Flat 8 (1)
left out, too few sales: Flat 9+ (3)


sales  median price  median floor area (m2)
property type habitable rooms                                             
Detached      3                  150        525000                    75.0
              4                  475        600000                    92.0
              5                  865        670000                   109.0
              6                  937        766000                   131.0
              7                  823        875000                   152.2
              8                  556       1052000                   185.0
              9+                 527       1305000                   243.0
Semi          2                   31        300000                    49.0
              3                  580        416000                    65.0
              4                 1866        491000                    84.6
              5                 3162        550000                    97.0
              6                 1574        640000                   120.0
              7                  860        725000                   141.0
              8                  341        800000                   163.0
              9+                 186        951000                   194.0
Terraced      2                  163        285000                    46.0
              3                 1679        390000                    68.3
              4                 3891        425000                    80.0
              5                 4140        485000                    92.0
              6                 1402        570000                   113.0
              7                  392        640000                   132.0
              8                   96        748000                   152.0
              9+                  39        950000                   194.0
Flat          1                  216        187000                    32.1
              2                 3237        245000                    46.2
              3                 5762        320000                    64.0
              4                 1521        378000                    78.0
              5                  317        462000                    95.0
              6                   42        625000                   114.5

In [20]:
by_borough = sales.groupby(["borough", "type"], observed=True).agg(
    sales=("price", "size"),
    median_price=("price", "median"),
    median_area=("floor_area", "median"),
    median_price_per_m2=("price_per_m2", "median"))
by_borough = finish(by_borough, ["borough", "property type"])
by_borough

sales  median price  median floor area (m2)  \
borough   property type                                                
Bromley   Detached        2679        860000                   141.0   
          Semi            4496        600000                   101.0   
          Terraced        4083        499000                    87.2   
          Flat            5257        331000                    62.2   
Croydon   Detached        1684        742000                   136.3   
          Semi            3067        538000                   102.0   
          Terraced        4715        430000                    86.0   
          Flat            5509        295000                    61.0   
Greenwich Detached         241        875000                   129.0   
          Semi            1309        565000                    99.0   
          Terraced        3390        487000                    88.9   
          Flat            5429        410000                    65.0   

                         median price per m2  
borough   property type                       
Bromley   Detached                      6166  
          Semi                          5880  
          Terraced                      5708  
          Flat                          5366  
Croydon   Detached                      5431  
          Semi                          5148  
          Terraced                      5000  
          Flat                          5000  
Greenwich Detached                      6494  
          Semi                          5617  
          Terraced                      5557  
          Flat                          6531

## 11. Save the results

In [21]:
comparison.round(4).to_csv(f"{RESULTS_DIR}/floor_area_comparison.csv")
tuning.round(1).to_csv(f"{RESULTS_DIR}/floor_area_tuning.csv", index=False)
by_group.to_csv(f"{RESULTS_DIR}/floor_area_by_group.csv")
data_quality.to_csv(f"{RESULTS_DIR}/floor_area_data_quality.csv", index=False)
split_sizes.to_csv(f"{RESULTS_DIR}/floor_area_split_sizes.csv", index=False)
match_check.round(1).to_csv(f"{RESULTS_DIR}/floor_area_match_check.csv")
settings_check.round(1).to_csv(f"{RESULTS_DIR}/floor_area_settings_check.csv")
by_size.to_csv(f"{RESULTS_DIR}/price_by_size.csv")
by_rooms.to_csv(f"{RESULTS_DIR}/price_by_rooms.csv")
by_borough.to_csv(f"{RESULTS_DIR}/price_per_m2.csv")
inputs.to_csv(f"{RESULTS_DIR}/floor_area_inputs.csv", index=False)

print("saved to", RESULTS_DIR)

saved to results


## References
- Chi, B., Dennett, A., Oléron-Evans, T. and Morphet, R. (2021) A new attribute-linked residential property price dataset for England and Wales, 2011–2019. *UCL Open: Environment*, 2021(2), Article 07. https://doi.org/10.14324/111.444/ucloe.000019
- Chi, B., Dennett, A., Oléron-Evans, T. and Morphet, R. (2025) *House Price Per Square Metre in England and Wales, 1995-2024* [data collection]. Colchester, Essex: UK Data Service. https://doi.org/10.5255/UKDA-SN-857911. The copy used here is the 2024 file on the London Datastore (maintainers Chi, Dennett, Oléron-Evans and Morphet, all UCL), whose source data was downloaded on 26 December 2024: https://data.london.gov.uk/dataset/house-price-per-square-metre-in-england-and-wales
- HM Land Registry, *Price Paid Data*. Contains HM Land Registry data © Crown copyright and database right 2021. This data is licensed under the Open Government Licence v3.0.
- Floor areas and room counts in the UCL file come from Domestic Energy Performance Certificates published by the Ministry of Housing, Communities and Local Government.